# 02 — Data cleaning

## Goal of this step

Translate audit findings into reproducible, tested cleaning rules. Save a clean table while preserving the raw CSV.

## Why this matters

Cleaning is like standardizing addresses before delivering mail: ' No ' and 'No' should mean the same thing. A rule based on the current row is safe before splitting; a learned median is not and belongs inside training folds.

Find the project root and import display tools. Reusable logic lives in src.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np  # noqa: F401
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
plt.rcParams["figure.figsize"] = (8, 4)


Inspect the problematic rows before changing them.

In [2]:
from src.data import load_raw, run_cleaning
raw = load_raw()
display(raw.loc[pd.to_numeric(raw['TotalCharges'], errors='coerce').isna(), ['tenure', 'TotalCharges']])

,tenure,TotalCharges
488,0,
753,0,
936,0,
1082,0,
1340,0,
3331,0,
3826,0,
4380,0,
5218,0,
6670,0,


**Interpretation:** All 11 blanks occur at tenure zero, supporting a cumulative-charge value of zero.

Run the shared cleaning function. It strips whitespace, removes customerID, converts charges, normalizes service labels, maps Churn, and drops duplicate cleaned records.

In [3]:
df, comparison = run_cleaning()
display(comparison)
display(pd.Series({'raw_rows': len(raw), 'clean_rows': len(df), 'remaining_nulls': int(df.isna().sum().sum())}))

,before_dtype,after_dtype,before_nulls,after_nulls
Churn,str,int64,0,0.0
Contract,str,str,0,0.0
Dependents,str,str,0,0.0
DeviceProtection,str,str,0,0.0
InternetService,str,str,0,0.0
MonthlyCharges,float64,float64,0,0.0
MultipleLines,str,str,0,0.0
OnlineBackup,str,str,0,0.0
OnlineSecurity,str,str,0,0.0
PaperlessBilling,str,str,0,0.0


raw_rows           7043
clean_rows         7021
remaining_nulls       0
dtype: int64

**Interpretation:** The dtype table shows numeric charges and a numeric target; 22 identical cleaned records are removed, leaving 7,021 rows.

Verify the conditional rule and remaining categories. np.where changes only missing charges at tenure zero; unknown bills at positive tenure remain missing for fold-local median imputation.

In [4]:
display(df.loc[df['tenure'].eq(0), ['tenure', 'TotalCharges']])
display(df[['MultipleLines', 'OnlineSecurity', 'TechSupport']].apply(lambda col: ', '.join(sorted(col.unique()))).to_frame('values'))
display(df['Churn'].value_counts().to_frame('customers'))

,tenure,TotalCharges
488,0,0.0
753,0,0.0
936,0,0.0
1081,0,0.0
1338,0,0.0
3323,0,0.0
3817,0,0.0
4370,0,0.0
5207,0,0.0
6652,0,0.0


,values
MultipleLines,"No, Yes"
OnlineSecurity,"No, Yes"
TechSupport,"No, Yes"


,customers
Churn,
0,5164
1,1857


**Interpretation:** No parent service means no add-on, while InternetService itself still identifies customers without internet.

## Common mistakes

- Replacing every invalid charge with zero regardless of tenure.
- Learning imputation medians from the full dataset.
- Dropping duplicate feature rows with different targets as though they were identical.

## Key takeaways

- The raw file remains unchanged.
- Row-local rules are explicit and covered by tests.
- No statistical transformer has been fit yet.

## Next step

03_eda.ipynb: split first and explore only training customers. Deduplication after dropping IDs is a coursework choice: distinct people can share identical attributes; production work should use entity-level deduplication instead.